The model in our paper was trained on proprietary data, hence the data are not public.
 
Here we generate some dummy Cell Painting (CP) and Transcriptomics (TX) data just to run the code.

If you have data and wants to know how to format it to run with the mode, also follow this notebook.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
CP_DIM = 1600 # Dimension of CP feature
TX_DIM = 18000 # Dimension of TX feature

TRAIN_SET_LEN = 70000 
VAL_SET_LEN =  10000
TEST_SET_LEN = 20000 

DATA_SET_LEN = TRAIN_SET_LEN + VAL_SET_LEN + TEST_SET_LEN

NUM_COMPOUNDS = 50000 # Lower than DATA_SET_LEN to simulate replicates

### Create .npy data files

CP and TX numerical values are stored in seperate .npy array matrices

In [3]:
cp_data = np.random.randn(DATA_SET_LEN, CP_DIM).astype(np.float32)
print(f"CP Data Dimension: {cp_data.shape}")
tx_data = np.random.randn(DATA_SET_LEN, TX_DIM).astype(np.float32)
print(f"CP Data Dimension: {tx_data.shape}")

CP Data Dimension: (100000, 1600)
CP Data Dimension: (100000, 18000)


### Create .csv index files

We then map the row index of the CP and TX values matrices to each other here

In [4]:
# Map rows of CP and TX matrices

np.random.seed(seed=42)
# Index for TX. In this work, TX is unique for each compound. In other situations we can also have TX replicates
list_tx_idx = list(np.random.randint(low=0, high=NUM_COMPOUNDS, size=DATA_SET_LEN)) 
# Unique compound identifiers
list_compound_idx = [f"C{i}" for i in list_tx_idx]
# Index for CP
list_cp_idx = list(range(DATA_SET_LEN))
np.random.shuffle(list_cp_idx)

# Create Pandas DataFrame
full_dataset = pd.DataFrame({
    "compound":list_compound_idx,
    'index_tx':list_tx_idx,
    "index_cp":list_cp_idx,
    "index_emb":0 # Index of learned embeddings, will be filled in later in this notebook
})

In [5]:
# Split train, val, test set

trainval_set, test_set = train_test_split(full_dataset, train_size=TRAIN_SET_LEN+VAL_SET_LEN, test_size=TEST_SET_LEN, random_state=42)
train_set, val_set = train_test_split(trainval_set, train_size=TRAIN_SET_LEN, test_size=VAL_SET_LEN, random_state=42)

train_set = train_set.reset_index(drop=True)
val_set = val_set.reset_index(drop=True)
test_set = test_set.reset_index(drop=True)

In [6]:
# Add values for index_emb

list_cp_idx = list(range(DATA_SET_LEN))

train_set['index_emb'] = list_cp_idx[:TRAIN_SET_LEN]
val_set['index_emb'] = list_cp_idx[TRAIN_SET_LEN:TRAIN_SET_LEN+VAL_SET_LEN]
test_set['index_emb'] = list_cp_idx[TRAIN_SET_LEN+VAL_SET_LEN:]

In [7]:
print(train_set.head())

  compound  index_tx  index_cp  index_emb
0   C42189     42189     67617          0
1   C26776     26776     60200          1
2   C21786     21786     14411          2
3   C48596     48596     63401          3
4   C24779     24779       806          4


### Save the files

In [8]:
# Save .npy files
np.save('../data/CP_values.npy', cp_data)
np.save('../data/TX_values.npy', tx_data)

# Save .csv files
train_set.to_csv('../data/train_set_index.csv', index=False)
val_set.to_csv('../data/val_set_index.csv', index=False)
test_set.to_csv('../data/test_set_index.csv', index=False)